# BigQuery: when the data is too big to download

Every API so far followed one shape. Ask for a thing, get the thing.

BigQuery breaks that shape, and the break is the point.

The London cycle hire table in this notebook is **9.6 GiB and 83 million
rows**. You cannot `requests.get` it. You would not want it on your laptop if
you could. Your practice database `bikeshare.db` holds 17,379 rides; this
table holds roughly **4,800 times** as many.

So instead of sending a request and receiving data, you **send the question**
and receive only the answer. The SQL travels. The data stays where it is and
is read by many machines at once.

Three things follow from that inversion, and they are the whole notebook:

1. You must **prove who you are**. There is no anonymous access.
2. You must **say who pays**. Reading is not free.
3. You pay for **what gets read**, not for what comes back.

### What you need first

```
pip install google-cloud-bigquery db-dtypes
gcloud auth application-default login
```

The second command opens a browser once. Section 1 explains what it leaves
behind and why that is the safe option.

---

## 1. Authentication: two different questions

People lose hours here because two separate questions look like one.

- **Who are you?** That is *authentication*.
- **Whose bill does this go on?** That is the *billing project*.

The answers are different. You are a person with a Google account. The bill
goes to a **project**, a container you create. The public dataset belongs to
neither of you: it sits in a project named `bigquery-public-data`, and
**reading it is free while scanning it is not**. Your project pays for the
machines that do the scanning.

### Why there is no API key

Some services earlier took `?api_key=...` on the end of a URL. BigQuery never
will, and the reason is worth understanding.

A key in a URL is a **bearer token** — whoever holds it is you. URLs get
logged by proxies, saved in browser history, and pasted into chat. That is
tolerable for a weather lookup. It is not tolerable for something that can
spend money and read private tables.

So Google issues short-lived tokens instead, refreshed automatically, and
gives you three ways to obtain them.

### The three ways, and which one to use

| Way | What it is | Use it when |
|---|---|---|
| **ADC, user credentials** | You log in once in a browser. A refresh token lands in your home directory, **outside any project folder** | On your own laptop. This is what we use |
| **Service account key file** | A `.json` file that *is* a password. No expiry, no login, works for whoever holds it | Almost never on a laptop |
| **Attached identity** | Code running on Google's own infrastructure is issued an identity with no file at all | Production, on GCP |

ADC stands for **Application Default Credentials**. You set it up once:

```
gcloud auth application-default login
```

A browser opens, you approve, and it is done. Nothing to copy, nothing to
paste, nothing to commit by accident.

> **The middle row is the one that causes damage.** A service account key is
> a plaintext credential with no expiry date. Leave it in a project folder
> and one `git add .` publishes it. Automated scanners search public
> repositories for exactly that file, and an unexpected bill can follow
> within hours.
>
> If an instruction anywhere tells you to download a key file into your
> working directory, use ADC instead. The safest key file is the one you
> never created.

### Look at your own credential

It is a small text file. Nothing mystical about it.

The cell below prints its **structure only**. Every secret value is replaced
by a character count, so the cell is safe to run with your screen shared.

In [ ]:
import json
import stat
from pathlib import Path

adc = Path.home() / ".config/gcloud/application_default_credentials.json"

print("file       :", adc)
print("exists     :", adc.exists())
print("permissions:", stat.filemode(adc.stat().st_mode))
print("size       :", adc.stat().st_size, "bytes")
print()

SAFE_TO_SHOW = {"type", "quota_project_id", "universe_domain"}

for key, value in sorted(json.loads(adc.read_text()).items()):
    if key in SAFE_TO_SHOW:
        print("  {:<18} {}".format(key, value))
    elif not str(value):
        print("  {:<18} <empty>".format(key))
    else:
        print("  {:<18} <{} characters, not printed>".format(key, len(str(value))))

Three things to take from that output.

**`permissions: -rw-------`** means owner read and write, nobody else, not
even other users on this machine. That is `chmod 600`, and it is deliberate.

**`type: authorized_user`** is the field that tells you which kind of
credential you are holding. A service account key file says
`type: service_account` instead. Whenever you find a Google JSON credential
and want to know how dangerous it is, read that field first.

**`refresh_token`** is not a password for BigQuery. It is used to fetch
*short-lived* access tokens, each valid for about an hour. Stolen, it is
still bad, but it can be revoked centrally. A service account key
effectively cannot.

Note where the file lives: `~/.config/gcloud/`. **Your home directory, not
your project.** No `git add` can reach it. That is not incidental. It is the
main reason to prefer ADC.

### Who the library thinks you are

`google.auth.default()` is the function every Google library calls to find
your credentials. It looks in order at the `GOOGLE_APPLICATION_CREDENTIALS`
environment variable, then the ADC file above, then, if the code is running
on GCP, the machine's attached identity.

In [ ]:
import google.auth
from google.cloud import bigquery

credentials, discovered_project = google.auth.default()

print("credential class :", type(credentials).__name__)
print("project it found :", discovered_project)

It found out *who you are* but very likely printed `None` for the project. It
did not find out *who pays*. Those are two separate questions, and that is
the proof.

So say it explicitly. Always be explicit here — a project picked up from
ambient configuration is a project you will eventually bill by accident.

In [ ]:
PROJECT = "dsai-teaching-sandbox"   # who pays. Change this to your own project.

client = bigquery.Client(project=PROJECT)

print("authenticated, billing project:", client.project)

> **Create your project with billing switched off.** With no billing account
> attached, BigQuery runs in **sandbox** mode: 1 TiB of query scanning per
> month, free, and when the allowance runs out queries simply fail. Nothing
> can be charged, because there is no card attached to charge.
>
> Watch for one thing. Google may attach a **new project to an existing
> billing account automatically**, without asking. Check under
> **Billing > Account management** and detach it.

---

## 2. Public datasets, and the free question

`bigquery-public-data` is a project full of real datasets that Google hosts
and anyone may read: census records, weather, patents, source code, bike
hires.

A table name has **three parts**, wrapped in backticks:

```sql
`project.dataset.table`
```

You are about to read tables in a project that is not yours. That is normal
here.

Start with the cheapest possible question. How big is it?

> **Predict first.** Counting 83 million rows. How many bytes do you think
> that has to read?

In [ ]:
tables = [
    ("London cycle hires", "bigquery-public-data.london_bicycles.cycle_hire"),
    ("Austin bikeshare  ", "bigquery-public-data.austin_bikeshare.bikeshare_trips"),
    ("USA baby names    ", "bigquery-public-data.usa_names.usa_1910_current"),
]

for label, table in tables:
    job = client.query("SELECT COUNT(*) AS n FROM `{}`".format(table))
    rows = list(job)
    print("{}  {:>13,} rows   billed: {} bytes".format(
        label, rows[0]["n"], job.total_bytes_billed))

**Zero bytes, three times.**

`COUNT(*)` with no filter is answered from the table's own bookkeeping.
BigQuery already knows how many rows it holds and never opens the data.

Free here does not mean free-because-small. It means free because **nothing
was read**. Hold on to that distinction, because the next section is built
on it.

For comparison: 83 million rows against `bikeshare.db`'s 17,379. The same
kind of data — bikes, stations, timestamps — and the SQL you already know
works on both without change.

In [ ]:
table = client.get_table("bigquery-public-data.london_bicycles.cycle_hire")

print("size on disk:", round(table.num_bytes / 2**30, 1), "GiB")
print("rows        :", format(table.num_rows, ","))
print("columns     :", len(table.schema))
print()

for field in table.schema[:6]:
    print("  {:<22} {}".format(field.name, field.field_type))

`get_table` reads metadata, so it is free as well.

**Read the schema before you write any SQL.** It is the BigQuery equivalent
of printing a dictionary's keys before using it.

---

## 3. The cost model, which is not the one you expect

One sentence governs everything:

> **You pay for the bytes BigQuery reads, not for the rows it gives back.**

BigQuery is **columnar**. Each column is stored separately, so a query reads
only the columns you name — but it reads those columns *whole*.

> **Predict first.** Five queries against the same six-million-row table.
> Rank them by cost before running the cell, and pay particular attention to
> the one with `LIMIT 10`.

A **dry run** asks what a query would cost without running it. It is free and
immediate.

In [ ]:
TABLE = "`bigquery-public-data.usa_names.usa_1910_current`"


def dry_run_bytes(sql):
    """Ask what a query would scan, without running it. Costs nothing."""
    settings = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=settings)
    return job.total_bytes_processed


questions = [
    ("one column",           "SELECT name FROM " + TABLE),
    ("two columns",          "SELECT name, number FROM " + TABLE),
    ("every column (*)",     "SELECT * FROM " + TABLE),
    ("one column, LIMIT 10", "SELECT name FROM " + TABLE + " LIMIT 10"),
    ("COUNT(*)",             "SELECT COUNT(*) FROM " + TABLE),
]

for label, sql in questions:
    scanned = dry_run_bytes(sql)
    print("{:<22} {:>12,} bytes   {:>6.1f} MiB".format(
        label, scanned, scanned / 2**20))

Read the fourth row again.

**`LIMIT 10` cost exactly the same as no limit at all.** The full column, to
hand back ten names.

This is the mistake behind the horror stories. In every database you have
used until now, `LIMIT` made a query cheap. Here it only shrinks the
*output*; the scan has already happened. Someone "just taking a quick look"
with `SELECT * FROM huge_table LIMIT 10` has paid for the entire table.

The other rows say the rest:

- **Two columns cost twice one column.** The column is the unit of cost.
- **`SELECT *` cost about four times one column.** You paid for 14 columns
  in order to use two.
- **`COUNT(*)` was free.** Metadata again.

So **name your columns, and never write `SELECT *`**. When you want to eyeball
the data, use the table preview in the BigQuery console, which is free,
rather than a `LIMIT` query, which is not.

### The seatbelt

A dry run only helps when you remember to do one. `maximum_bytes_billed` is
the protection for when you forget: BigQuery refuses outright any query that
would exceed it. Set it on every job.

> **Predict first.** We put a 1 MB ceiling on a `SELECT *`. What kind of
> error comes back?

In [ ]:
from google.api_core.exceptions import GoogleAPICallError

seatbelt = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000,   # 1 MB
                                   use_query_cache=False)

try:
    job = client.query("SELECT * FROM " + TABLE + " LIMIT 5", job_config=seatbelt)
    list(job)
    print("it ran, billed:", job.total_bytes_billed)
except GoogleAPICallError as problem:
    print("caught:", type(problem).__name__)
    print(str(problem).split("Location:")[0].strip())

Two things happened there. One is useful, one is a wart.

**The useful part:** the refusal names the real number, `... or higher
required`. That is the cost of returning five rows through a `SELECT *`. The
seatbelt has just told you the query was far more expensive than it looked.

**The wart:** it arrives as **`InternalServerError`, a 500**. By the usual
rule — 4xx is your fault, 5xx is theirs — this should be a `403`. You set a
limit and you hit it; nothing broke on Google's side. It is misclassified,
which means a `try/except` written to catch `Forbidden` will miss it
entirely.

Catch `GoogleAPICallError`, which is the parent of both. When documentation
and observed behaviour disagree, believe the behaviour.

### The cache will fool you

Run identical SQL twice and the second run is free. The result was
remembered for 24 hours.

In [ ]:
SQL = """
SELECT name, SUM(number) AS births
FROM `bigquery-public-data.usa_names.usa_1910_current`
WHERE year = 1999
GROUP BY name
ORDER BY births DESC
LIMIT 3
"""

for attempt in [1, 2]:
    job = client.query(SQL)
    list(job)
    print("run {}: billed {:>12,} bytes   cache_hit: {}".format(
        attempt, job.total_bytes_billed, job.cache_hit))

Free the second time. **Identical** means byte-for-byte identical SQL —
change one space and you pay again.

Excellent for your bill. A menace when you are *measuring* anything, because
the second run reports zero and you conclude the query is cheap. Every cell
in this notebook that measures cost passes `use_query_cache=False` for
exactly that reason.

---

## 4. A real question

Now use it for what it is for: a question you could not answer on your own
machine, because the data does not fit on it.

Which London stations start the most hires, and how long do those rides last?

Dry run first. Always.

In [ ]:
SQL = """
SELECT
  start_station_name,
  COUNT(*) AS hires,
  ROUND(AVG(duration) / 60, 1) AS avg_minutes
FROM `bigquery-public-data.london_bicycles.cycle_hire`
GROUP BY start_station_name
ORDER BY hires DESC
LIMIT 8
"""

would_scan = dry_run_bytes(SQL)

print("would scan:", round(would_scan / 2**30, 2), "GiB")
print("at $6.25 per TiB, about $", round(would_scan / 2**40 * 6.25, 4))
print("that is", round(would_scan / 2**40 * 100, 2), "% of the 1 TiB monthly free tier")

In [ ]:
settings = bigquery.QueryJobConfig(maximum_bytes_billed=4 * 2**30,   # 4 GiB ceiling
                                   use_query_cache=False)

job = client.query(SQL, job_config=settings)
busiest = job.to_dataframe()

print("billed       :", format(job.total_bytes_billed, ","), "bytes")
print("rows returned:", len(busiest))
print()
print(busiest.to_string(index=False))

Close to 3 GiB read. Eight rows returned.

That is the cost model in a single observation: **the size of the answer told
you nothing about the size of the bill.**

The answer itself is worth reading properly. Sort the rows into two groups by
their average ride length:

- **Hyde Park Corner, Albert Gate, Black Lion Gate, Wellington Arch** — well
  over half an hour
- **King's Cross, Waterloo** — around a quarter of an hour

Parks and railway stations. The park stations are people riding in a loop for
the sake of it. The rail stations are commuters going somewhere specific.
**Two different populations sharing one bike scheme, separated by a single
`AVG`** — and the station names are doing the work of a column this dataset
does not contain.

That finding needs all 83 million rows, and it cost about two cents.

> **A warning you may see.** `to_dataframe()` can print `BigQuery Storage
> module not found`. It is harmless — the library fell back to a slower path.
> Silence it with `pip install google-cloud-bigquery-storage`, or use
> `list(job)` and build the DataFrame yourself.

---

## What to take away

### On authentication

1. **Two questions, not one.** Who you are, and who pays. Set the project
   explicitly every time.
2. **No API keys here.** A key in a URL gets logged. BigQuery uses
   short-lived tokens instead.
3. **ADC on a laptop.** `gcloud auth application-default login` leaves a
   revocable credential in your home directory, where git cannot reach it.
4. **A service account key file is a plaintext password with no expiry.** If
   any instruction tells you to download one into your project folder, use
   ADC instead.
5. **The `type` field tells you what you are holding** — `authorized_user` or
   `service_account`.
6. **Create your project with billing off.** Sandbox mode gives 1 TiB a month
   and cannot charge you. Check that a billing account has not been attached
   for you.

### On querying

7. **Send the question, not the request.** The data is too big to move. Your
   SQL is not.
8. **You pay for bytes read, not rows returned.** Around 3 GiB scanned to
   return 8 rows.
9. **`LIMIT` does not make a query cheap.** It shrinks the output after the
   scan. This is the expensive surprise.
10. **Name your columns.** `SELECT *` cost roughly four times the two columns
    that were needed.
11. **Metadata is free** — `COUNT(*)`, `get_table`, dry runs. Use them before
    anything else.
12. **Dry run, then set `maximum_bytes_billed`.** The first is discipline; the
    second is the seatbelt for when discipline lapses.
13. **Beware the cache when measuring.** Identical SQL bills zero the second
    time and will convince you an expensive query is cheap.

### Try these yourself

- Dry-run `SELECT * FROM bigquery-public-data.london_bicycles.cycle_hire`.
  How much of your monthly allowance would that one query use?
- Add `WHERE EXTRACT(YEAR FROM start_date) = 2016` to the station query and
  dry-run it again. Did filtering the rows reduce the bytes? Explain what you
  see.
- Find your own name in `usa_names` and plot it by year. Name only the
  columns you need, and check the cost before you run it.
- Run `get_table` on three other public tables and read the schema before
  writing a line of SQL.